# UTA-RLDD training (Colab / cloud) — resumable

Trains the feature baseline and the small GRU with the repository's own pipeline (`python -m research ...`).

**Honest status:** this notebook has *not* been run on the real dataset by the author of this repo. It has no outputs. Every number it produces comes from *your* run.

Prerequisites: the repo folder on Drive (or git clone), and the UTA-RLDD videos on Drive/local disk (raw videos stay private; nothing here uploads them).
Everything is resumable: extraction skips finished videos; GRU training checkpoints every epoch to `ARTIFACTS`. Put `ARTIFACTS` on Drive so a disconnect loses nothing.

## 1. Paths (EDIT THESE)

In [ ]:
# --- explicit dataset / repo / output paths -------------------------------------------------
REPO_DIR      = "/content/drive/MyDrive/MBU_PROJECT"          # folder containing app.py, dms/, research/
DATASET_ROOT  = "/content/drive/MyDrive/UTA-RLDD"             # root of the downloaded dataset (read-only use)
ARTIFACTS     = "/content/drive/MyDrive/rldd_artifacts"       # manifests, features, runs, logs (persist this!)
CONFIG        = f"{REPO_DIR}/configs/my_rldd.json"            # copy configs/default.json and fill layout_regex + label_source
RUN_NAME      = "run1"
WORKERS       = 2                                             # extraction processes; match your runtime's CPU count
USE_DRIVE     = True

In [ ]:
import os, sys, subprocess, pathlib
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
assert pathlib.Path(REPO_DIR, "research").is_dir(), "REPO_DIR is wrong"
assert pathlib.Path(DATASET_ROOT).is_dir(), "DATASET_ROOT is wrong"
os.chdir(REPO_DIR)
def run(*args):
    cmd = [sys.executable, "-m", "research", "--config", CONFIG, "--artifacts", ARTIFACTS, "--dataset-root", DATASET_ROOT, *args]
    print("$", " ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
!pip -q install -r requirements-research.txt
!python setup_model.py     # explicit one-time download of the official Face Landmarker model (needs internet)
import torch; print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Inspect the real files, then write the config

`inspect` prints what is on disk and assigns nothing. Compare the path signatures with the dataset's own documentation, then fill `dataset.layout_regex` (named groups `fold`, `participant`) and `dataset.label_source` in `CONFIG` (see `DATASET.md`). Filename-stem labels require `confirmed: true` plus a citation — set it only after *you* verified it.

In [ ]:
run('inspect')

## 3. Manifest (participant, video, label, duration, split) and leakage check

In [ ]:
run('manifest')
run('verify-splits')

## 4. Feature extraction (MediaPipe, resumable — safe to re-run after a disconnect)
Smoke first: `run('extract','--max-videos','2','--max-seconds','60')`, then the full run.

In [ ]:
run('extract', '--max-videos', '2', '--max-seconds', '60')   # smoke test; delete this line for the full run if satisfied

In [ ]:
run('extract', '--workers', str(WORKERS))

## 5. Heuristic (existing app logic) evaluation where comparable

In [ ]:
run('heuristic')

## 6. Feature baseline (train/validation only)

In [ ]:
run('train-baseline', '--run', RUN_NAME + '_baseline')

## 7. Small GRU (resumable; checkpoints in `ARTIFACTS/runs/<run>/gru/`)
Re-running this cell resumes from `last.pt`. Use `--no-resume` to start over.

In [ ]:
run('train-gru', '--run', RUN_NAME + '_gru')

## 8. Held-out evaluation — look at the TEST split deliberately and rarely
Each call is recorded in `eval_log.jsonl`; repeated looks are flagged in the report because they invite test-set tuning. Decide which model to evaluate on **validation** results first.

In [ ]:
run('evaluate', '--run', RUN_NAME + '_baseline', '--split', 'val')
run('evaluate', '--run', RUN_NAME + '_gru', '--split', 'val')

In [ ]:
CHOSEN = RUN_NAME + '_baseline'   # choose from the VALIDATION results above, not from test
run('evaluate', '--run', CHOSEN, '--split', 'test')
print(open(f'{ARTIFACTS}/runs/{CHOSEN}/report_test.md').read())

## 9. Export the bundle for the Windows app
Download `models_learned/` and place it at `models/learned/` in the app folder (or pass `--learned-model`).

In [ ]:
run('export', '--run', CHOSEN, '--dest', f'{ARTIFACTS}/models_learned', '--require-heldout')